In [3]:
import os
import joblib
import numpy as np
import pandas as pd
import json
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import classification_report, precision_recall_curve
from xgboost import XGBClassifier

try:
    print("Iniciando pipeline...")

    # USANDO O DATASET BALANCEADO
    caminho_dataset = os.path.join("..", "output", "dataset_unificado_balanceado.csv")
    df = pd.read_csv(caminho_dataset)
    print("Dataset balanceado carregado!")
    print(f"Formato: {df.shape}\n")

    df['requisitos_vaga'] = df['requisitos_vaga'].fillna('').str.lower()
    df['cv_texto'] = df['cv_texto'].fillna('').str.lower()

    print("Calculando similaridade textual...")
    todos_textos = pd.concat([df['requisitos_vaga'], df['cv_texto']])
    vetorizador_sim = TfidfVectorizer(max_features=300)
    vetorizador_sim.fit(todos_textos)

    req_matrix = vetorizador_sim.transform(df['requisitos_vaga'])
    cv_matrix = vetorizador_sim.transform(df['cv_texto'])
    df['sim_textual'] = np.array(req_matrix.multiply(cv_matrix).sum(axis=1)).ravel()
    print("Similaridade textual calculada.\n")

    print("Criando features auxiliares...")
    df['match_nivel'] = (df['nivel_profissional_vaga'] == df['nivel_profissional_candidato']).astype(int)
    df['match_ingles'] = (df['nivel_ingles_vaga'] == df['nivel_ingles_candidato']).astype(int)
    df['match_profissional'] = (df['nivel_profissional_vaga'] == df['nivel_profissional_candidato']).astype(int)
    df['match_espanhol'] = (df['nivel_espanhol_vaga'] == df['nivel_espanhol_candidato']).astype(int)
    df['match_local'] = (df['local_vaga'] == df['local_candidato']).astype(int)
    df['match_academico'] = (df['nivel_academico_vaga'] == df['nivel_academico_candidato']).astype(int)

    if 'match' not in df.columns:
        raise ValueError("Coluna 'match' não encontrada no dataset.")

    colunas_vazamento = ['situacao', 'comentario', 'recrutador']
    X = df.drop(columns=['match'] + colunas_vazamento)
    y = df['match']

    cat_cols = [
        'titulo_vaga', 'nivel_profissional_vaga',
        'nivel_ingles_vaga', 'nivel_espanhol_vaga', 'nivel_academico_vaga',
        'nivel_academico_candidato', 'nivel_ingles_candidato',
        'nivel_espanhol_candidato', 'nivel_profissional_candidato',
        'local_candidato', 'cliente', 'local_vaga'
    ]

    num_cols = [
        'sim_textual', 'match_nivel', 'match_ingles',
        'match_profissional', 'match_espanhol', 'match_local',
        'match_academico'
    ]

    colunas_utilizadas = [col for col in cat_cols + num_cols + ['requisitos_vaga', 'cv_texto'] if col in X.columns]
    X = X[colunas_utilizadas]

    print("\nDistribuição do target:")
    print(y.value_counts(normalize=True).rename("proportion"))

    num_pipeline = Pipeline([
        ('imputer', SimpleImputer(strategy='mean')),
        ('scaler', StandardScaler())
    ])

    cat_pipeline = Pipeline([
        ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
        ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=True))
    ])

    text_pipeline = TfidfVectorizer(max_features=30)

    cat_cols_validos = [col for col in cat_cols if col in X.columns]
    num_cols_validos = [col for col in num_cols if col in X.columns]

    preprocessor = ColumnTransformer(
        transformers=[
            ('num', num_pipeline, num_cols_validos),
            ('cat', cat_pipeline, cat_cols_validos),
            ('req_text', text_pipeline, 'requisitos_vaga'),
            ('cv_text', text_pipeline, 'cv_texto')
        ],
        sparse_threshold=1.0  # Usa matriz esparsa sempre que possível
    )

    X_train_raw, X_test, y_train, y_test = train_test_split(
        X, y, stratify=y, test_size=0.2, random_state=42
    )

    total_pos = y_train.sum()
    scale_pos_weight = (len(y_train) - total_pos) / total_pos
    print(f"\nscale_pos_weight: {scale_pos_weight:.2f}")

    print("Transformando dados...")
    X_train_transf = preprocessor.fit_transform(X_train_raw)
    X_test_transf = preprocessor.transform(X_test)

    print("Treinando modelo XGBoost...")
    modelo = XGBClassifier(
        n_estimators=100,
        eval_metric='logloss',
        scale_pos_weight=scale_pos_weight,
        use_label_encoder=False,
        early_stopping_rounds=10,
        random_state=42
    )
    modelo.fit(
        X_train_transf, y_train,
        eval_set=[(X_test_transf, y_test)],
        verbose=False
    )
    print("Modelo treinado.\n")

    y_pred_default = modelo.predict(X_test_transf)
    print("Avaliação (threshold padrão):")
    print(classification_report(y_test, y_pred_default, digits=3))

    print("\nAjuste por grupo (nivel_profissional_vaga) com foco em recall ≥ 0.6 e precisão ≥ 0.25...")
    y_probs = modelo.predict_proba(X_test_transf)[:, 1]
    X_test_com_probs = X_test.copy()
    X_test_com_probs['prob'] = y_probs
    X_test_com_probs['true'] = y_test.values

    thresholds_dinamicos = {}

    for grupo in X_test_com_probs['nivel_profissional_vaga'].unique():
        grupo_mask = X_test_com_probs['nivel_profissional_vaga'] == grupo
        y_true_grupo = X_test_com_probs.loc[grupo_mask, 'true']
        y_probs_grupo = X_test_com_probs.loc[grupo_mask, 'prob']

        if len(y_true_grupo) < 30:
            continue

        precs, recs, thrs = precision_recall_curve(y_true_grupo, y_probs_grupo)

        valid_idxs = [
            i for i in range(len(thrs))
            if recs[i] >= 0.6 and precs[i] >= 0.25
        ]

        if valid_idxs:
            best_idx = max(valid_idxs, key=lambda i: precs[i])
            thresholds_dinamicos[grupo] = float(thrs[best_idx])
        else:
            thresholds_dinamicos[grupo] = 0.5

    y_pred_dinamico = [
        int(prob >= thresholds_dinamicos.get(grupo, 0.5))
        for grupo, prob in zip(X_test_com_probs['nivel_profissional_vaga'], X_test_com_probs['prob'])
    ]

    print("\nAvaliação com threshold dinâmico (prioridade para recall ≥ 0.6 e precisão ≥ 0.25):")
    print(classification_report(y_test, y_pred_dinamico, digits=3))

    output_dir = os.path.join("..", "output")
    os.makedirs(output_dir, exist_ok=True)

    joblib.dump(modelo, os.path.join(output_dir, "modelo_match_xgb.joblib"))
    joblib.dump(vetorizador_sim, os.path.join(output_dir, "vetorizador_sim_textual.joblib"))
    joblib.dump(preprocessor, os.path.join(output_dir, "preprocessador_xgb.joblib"))

    with open(os.path.join(output_dir, "thresholds_dinamicos.json"), "w", encoding="utf-8") as f:
        json.dump(thresholds_dinamicos, f)

    print("\nExecução finalizada com sucesso!")
    print(f"Registros: {len(df)} | Positivos: {int(y.sum())} | Negativos: {int((~y.astype(bool)).sum())}")
    print("Artefatos salvos em:", output_dir)

except Exception as e:
    print(f"\nErro durante a execução: {str(e)}")


Iniciando pipeline...
Dataset balanceado carregado!
Formato: (10643, 22)

Calculando similaridade textual...
Similaridade textual calculada.

Criando features auxiliares...

Distribuição do target:
match
0    0.699991
1    0.300009
Name: proportion, dtype: float64

scale_pos_weight: 2.33
Transformando dados...
Treinando modelo XGBoost...


c:\Users\maiar\AppData\Local\Programs\Python\Python313\Lib\site-packages\xgboost\callback.py:386: UserWarning: [20:07:33] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  self.starting_round = model.num_boosted_rounds()


Modelo treinado.

Avaliação (threshold padrão):
              precision    recall  f1-score   support

           0      0.836     0.840     0.838      1490
           1      0.623     0.615     0.619       639

    accuracy                          0.773      2129
   macro avg      0.729     0.728     0.728      2129
weighted avg      0.772     0.773     0.772      2129


Ajuste por grupo (nivel_profissional_vaga) com foco em recall ≥ 0.6 e precisão ≥ 0.25...

Avaliação com threshold dinâmico (prioridade para recall ≥ 0.6 e precisão ≥ 0.25):
              precision    recall  f1-score   support

           0      0.832     0.797     0.814      1490
           1      0.568     0.624     0.595       639

    accuracy                          0.745      2129
   macro avg      0.700     0.711     0.704      2129
weighted avg      0.753     0.745     0.748      2129


Execução finalizada com sucesso!
Registros: 10643 | Positivos: 3193 | Negativos: 7450
Artefatos salvos em: ..\output
